# Publisher Application

This application makes use of the Confluence Publisher class defined in this folder. See ./Publisher.py

In [ ]:
import markupsafe
header=markupsafe.Markup('<span style="color:red;"><b>Unreliable information. For testing purposes only!</b></span><br/>')

## Load configuration
Be aware not to commit your credentials!

In [ ]:
import logging
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

LOGFILE = 'target/debug.log'

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
handler = logging.handlers.RotatingFileHandler(
    LOGFILE, maxBytes=(1048576*5), backupCount=7
)
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info('Starting execution')

In [ ]:
configuration_file = 'pim_model.yaml'

In [ ]:
import yaml
import copy
import logging
log = logging.getLogger(__name__)

with open(configuration_file) as f:
    config = yaml.safe_load(f)

conf_conf = config['confluence']
assert conf_conf
assert len(conf_conf['apiurl']) > 0
space_key = conf_conf['space']
root_page = conf_conf['rootpage']

conf_confidential = copy.deepcopy(config)
conf_confidential['confluence']['password'] = '***'
conf_confidential

In [ ]:
# enable overwrite by test infrastructure
confluence_username = config['confluence']['username']
confluence_password = config['confluence']['password']
cooldown = config['confluence'].get('cooldown', 0.0)
cooldown

## Load the data
The **data** is the JSON serialized information model 

In [ ]:
import json

data = None
with open(config['json'], 'r') as source:
     data = json.load(source)

str(data)[:512]

In [ ]:
categories = list(data)
for category in categories:
    print('{} Elements in category "{}"'.format(len(data[category]), category))

The [Confluence API](https://github.com/atlassian-api/atlassian-python-api) is embedded as a **git submodule** in the 'lib' folder next to this notebook.

Use `git submodule update --init` to fetch all submodules after a checkout without `--recursive` option.

If the next cell fails, install confluence-api submodule from the repository root with:
`git submodule add -f https://github.com/atlassian-api/atlassian-python-api.git notebooks/contentfactory/lib/atlassian-python-api`

In [ ]:
import sys
import os

library = 'lib/atlassian-python-api'
sys.path.insert(0, os.path.abspath(library))
from atlassian import Confluence

In [ ]:
confluence = Confluence(url=config['confluence']['apiurl'], username=confluence_username, password=confluence_password, cloud=True)
root_page_id = confluence.get_page_id(space_key, config['confluence']['rootpage'])
root_page_id

In [ ]:
default_language = config['languages'][0]
language_page_root = confluence.get_page_id(space_key, '')

In [ ]:
import Publisher as cp
publisher = cp.Publisher(config, data, confluence, space_key, root_page_id, default_language)
list(map(lambda e: (e, publisher.translate(data['entities'][e]['name'])), list(data['entities'])[:3]))

In [ ]:
result = publisher.scan_current_content()
str(result)[:200]

# Publishing

## Prepare destination structure

In [ ]:
from tqdm.autonotebook import tqdm
from tqdm.notebook import tqdm_notebook
import time

total = len(publisher.content_map)
with tqdm_notebook(total=total, dynamic_ncols=True, unit='Page') as pbar:
    topic_dict = publisher.collect_recursive(config)
    topics = list(topic_dict)
    print('Processing topics ' + str(topics))
    for topic in topics:
        print('Processing ' + topic)
        entry_config = topic_dict[topic]
        
        element_title = topic
        if entry_config.get('title'):
            element_title = entry_config['title']

        folder_page_id = publisher.stub(element_title, root_page_id, labels=['topic','im-parent', 'im'])
        print('Created group page "{}". Confluence page id = {}'.format(element_title, str(folder_page_id['id'])))

        item_filter = entry_config.get('filter')
        
        for item_key in data[topic]:
            item = data[topic][item_key]

            if item.get('current_confluence_content'):
                pbar.update(1)
                continue
            
            if item_filter:
                filter_result = eval(item_filter)
                if not filter_result:
                    log.warning('Skipping {key} filtered out by {item_filter}'.format(key=item_key, item_filter=item_filter))
                    pbar.update(1)
                    item['filtered'] = True
                    continue
            
            pbar.set_description('Preparing element {}:{}'.format(topic, item_key))
            title = publisher.page_title(item_key)

            parent = folder_page_id['id']

            # hack: attribute nested under entity
            if topic == 'attributes':
                parent = publisher.page_for_key(item['entity'])['pageid']

            # hack: table nested under system
            if topic == 'tables':
                parent = publisher.page_for_key(item['interface-id+'])['pageid']

            # hack: column nested under table
            if topic == 'columns':
                parent = publisher.page_for_key(item['table-id'])['pageid']

            retry = 10
            while retry > 0:
                try:
                    create_result = publisher.stub(title, parent, labels=entry_config.get('labels'))
                    page_id = create_result['id']
                    item['current_confluence_content'] = create_result['current']
                    publisher.register_page_id(item_key, page_id)
                    retry = 0
                except Exception as e:
                    log.error('Failed to stub element {}:{}'.format(topic, item), e)
                    time.sleep(4.5)
                    retry -= 1

            pbar.update(1)
            time.sleep(cooldown)

## Now generate content

In [ ]:
str(publisher.content_map)[:512]

In [ ]:
from requests.exceptions import HTTPError
from colorama import Fore, Style

def print_http_error_details(e: HTTPError):
    print(Fore.RED + e.response.content.decode('utf-8'))
    from pprint import pprint
    print(Fore.YELLOW + str(vars(e)))
    pprint(vars(e.response))
    result = e.response.raw
    pprint(vars(result))
    print(Style.RESET_ALL)

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape
import re
import time

update_message = os.environ.get('UPDATE_MESSAGE', 'Automated update')

jinja_env = Environment(
    loader=FileSystemLoader('./templates'),
    autoescape=select_autoescape(['html', 'xml'])
)
jinja_env.globals.update({ 'util': publisher, 'header': header, 'update_message': update_message, 'jinja_env': jinja_env })

total = len(publisher.content_map)
with tqdm_notebook(total=total, dynamic_ncols=True, unit='Page') as pbar:

    topic_dict = publisher.collect_recursive(config)
        
    topics = list(topic_dict)
    for topic in topics:
        
        items = list(data[topic])    
        print('Creating content for class "{}" ({})'.format(topic, len(items)))
        topic_config = topic_dict[topic]

        jinja_template = jinja_env.get_template(topic_config['template'])

        for item_key in items:
            item = data[topic][item_key]
            
            if item.get('filtered'):
                pbar.update(1)
                continue
            
            if not item.get('icon'):
                item['icon'] = '0612'  # default icon
            
            try:
                rendered = jinja_template.render(data=data, key=item_key, item=item, config=topic_config)
                content_xml = re.sub('<!--.+?->(\n+)*', '', rendered) # strip comment lines
                
                minor_edit = False
                current = item.get('current_confluence_content')
                if current and current == content_xml:
                    minor_edit = True
                    pbar.set_description('Major update on element {}:{}'.format(topic, item_key))
                else:
                    pbar.set_description('Minor update on element {}:{}'.format(topic, item_key))
                
                retries = 10
                page = publisher.page_for_key(item_key)
                while retries > 0:
                    try:
                        result = publisher.update_page(item_key, content_xml, minor_edit=minor_edit, version_comment=update_message)
                        item['upload_successfull'] = True
                        item['confluence_update_result'] = result
                        retries = -10
                    except HTTPError as error:
                        log.error('Failed to update page id {id}: {title}'.format(id=page['pageid'], title=page['title']))
                        print_http_error_details(error)
                    except ConnectionError:
                        log.excpetion('Failed to publish, retry in 1s')
                        time.sleep(5.5)
                    finally:
                        retries -= 1
                        
                    if retries == 0:
                        item['upload_failed'] = True
                        log.error('Unable to update page id {id}: {title}'.format(id=page['pageid'], title=page['title']))
                        
            except Exception as e:
                log.exception('Unable to process {} {}'.format(topic, item_key))
                
            pbar.update(1)
            time.sleep(cooldown)

In [ ]:
children = confluence.get_page_child_by_type(root_page_id, limit=1000000)
len(children)

In [ ]:
with open('target/published.yaml', 'w') as dest:
    yaml.dump(data, dest)